# OULAD Graph Pipeline — Canonical Analysis Notebook

**Purpose**: Reproducible record of the Week 8 leakage-safe enrollment-centric graph.

This notebook:
1. Runs the full graph pipeline via `src/graph_pipeline.py`
2. Displays the integrity validation summary
3. Demonstrates `random_student_split` and `lcpo_split` on the enrollment supervision table

> **Note**: GNN training (GraphSAGE) and comparative evaluation are left for the following week.

**Target definition** (fixed throughout this project):
- `1 = at-risk` → Fail or Withdrawn (positive class)
- `0 = success` → Pass or Distinction
- All Precision, Recall, F1, AUPRC values refer to the at-risk class.


## 0. Setup


In [ ]:
import sys
from pathlib import Path

# Make src/ importable from the notebooks/ directory
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from graph_pipeline import run_pipeline
from oulad_data import random_student_split, lcpo_split
from config import GRAPH_ARTIFACTS_DIR, GRAPH_VALIDATION_DIR

print('Project root:', PROJECT_ROOT)
print('Artifacts dir:', GRAPH_ARTIFACTS_DIR)


## 1. Run the Week 8 Graph Pipeline

Runs all 7 stages: load → filter (cutoff 56 days) → nodes → edges → enrollments → validate → persist.
Assessment filtering uses **due date ≤ 56**, not submission date.


In [ ]:
result = run_pipeline(
    week=8,
    data_dir=str(PROJECT_ROOT / 'data' / 'raw'),
    save_dir=str(GRAPH_ARTIFACTS_DIR),
)


## 2. Graph Statistics


In [ ]:
nodes = result['nodes']
edges = result['edges']
enrollments = result['enrollments']

print('=== NODE COUNTS ===')
for ntype, ndf in nodes.items():
    print(f'  {ntype}: {len(ndf):,}')

print('\n=== EDGE COUNTS ===')
for etype, edf in edges.items():
    print(f'  {etype}: {len(edf):,}')

print(f'\n=== ENROLLMENTS ===')
print(f'  Total:   {len(enrollments):,}')
at_risk = int((enrollments["target"] == 1).sum())
print(f'  At-risk: {at_risk:,} ({at_risk/len(enrollments)*100:.1f}%)')
print(f'  Success: {len(enrollments)-at_risk:,} ({(len(enrollments)-at_risk)/len(enrollments)*100:.1f}%)')

print(f'\n=== PERFORMANCE ===')
print(f'  Elapsed:     {result["elapsed_seconds"]:.1f}s')
print(f'  Peak memory: {result["peak_memory_mb"]} MB')


## 3. Validation Summary

Read and display the integrity validation report generated by the pipeline.


In [ ]:
validation_summary = GRAPH_VALIDATION_DIR / 'week08_validation_summary.txt'
if validation_summary.exists():
    print(validation_summary.read_text())
else:
    print('Integrity checks from pipeline run:')
    for k, v in result['integrity'].items():
        flag = ' \u26a0' if isinstance(v, int) and k.startswith(('dup_', 'dangling_', 'null_')) and v > 0 else ''
        print(f'  {k}: {v}{flag}')


## 4. Label Distribution by Course-Presentation


In [ ]:
label_dist = (
    enrollments.groupby(['code_module', 'code_presentation'])['target']
    .agg(total='count', at_risk='sum')
    .assign(at_risk_rate=lambda d: (d['at_risk'] / d['total']).round(3))
    .sort_values('at_risk_rate', ascending=False)
    .reset_index()
)
print(label_dist.to_string(index=False))


## 5. random_student_split Demo

Demonstrates `random_student_split` from `src/oulad_data.py` on the Week 8 enrollment supervision table.

- Splits are on **unique students** — the same student cannot appear in more than one partition.
- Default fractions: val=10%, test=20%, train=70% of unique students.


In [ ]:
train_mask, val_mask, test_mask = random_student_split(
    enrollments, val_frac=0.1, test_frac=0.2, seed=42
)

train_enroll = enrollments[train_mask]
val_enroll   = enrollments[val_mask]
test_enroll  = enrollments[test_mask]

print('=== RANDOM STUDENT SPLIT (enrollment rows) ===')
print(f'  Train: {len(train_enroll):,} rows | {train_enroll["id_student"].nunique():,} unique students')
print(f'  Val:   {len(val_enroll):,} rows | {val_enroll["id_student"].nunique():,} unique students')
print(f'  Test:  {len(test_enroll):,} rows | {test_enroll["id_student"].nunique():,} unique students')

# Verify no student overlap between train and test
train_students = set(train_enroll['id_student'].unique())
test_students  = set(test_enroll['id_student'].unique())
assert train_students.isdisjoint(test_students), 'Student overlap detected!'
print('\n\u2713 No student overlap between train and test.')

# At-risk rates per split
for name, df in [('Train', train_enroll), ('Val', val_enroll), ('Test', test_enroll)]:
    rate = df['target'].mean()
    print(f'  At-risk rate \u2014 {name}: {rate:.3f}')


## 6. LCPO Split Demo

Demonstrates `lcpo_split` from `src/oulad_data.py`.

- For each of the 22 course-presentations, all enrollments in that presentation are held out as test; the rest are train.
- Verifies that all 22 splits yield non-empty train and test sets.


In [ ]:
presentations = (
    enrollments[['code_module', 'code_presentation']]
    .drop_duplicates()
    .sort_values(['code_module', 'code_presentation'])
    .reset_index(drop=True)
)

print(f'Course-presentations: {len(presentations)}')
print(f'{"-"*60}')
print(f'{"Module":<12} {"Presentation":<14} {"Train":>8} {"Test":>8} {"Test at-risk%":>14}')
print(f'{"-"*60}')

for _, row in presentations.iterrows():
    train_m, test_m = lcpo_split(enrollments, row.code_module, row.code_presentation)
    test_rate = enrollments.loc[test_m, 'target'].mean()
    print(
        f'{row.code_module:<12} {row.code_presentation:<14}'
        f' {train_m.sum():>8,} {test_m.sum():>8,} {test_rate*100:>13.1f}%'
    )

print(f'\n\u2713 All {len(presentations)} course-presentations yield non-empty train and test splits.')


## 7. Node Feature Null Audit

Confirms that all node feature tables have zero nulls after the imputation applied in `build_node_tables()`.


In [ ]:
print('Post-imputation null counts per node type:')
for ntype, ndf in nodes.items():
    feat_cols = [c for c in ndf.columns if c != 'node_idx']
    nulls = ndf[feat_cols].isnull().sum()
    total = int(nulls.sum())
    status = '\u2713' if total == 0 else f'\u26a0 {total} nulls'
    print(f'  {ntype}: {status}')
    if total > 0:
        print('   ', nulls[nulls > 0].to_dict())


---

*Notebook complete. GNN training (GraphSAGE, random-student and LCPO evaluation) follows in the next iteration.*
